# 🔬 Garby Detection Engine — Training Notebook

**Purpose:** Train a meta-classifier on top of the 5-layer Garby engine to learn optimal decision boundaries from real data.

## Setup checklist
1. Upload `garby_layer*.py` + `garby_orchestrator.py` to `/MyDrive/garby_engine/`
2. Create dataset folders:
   - `/MyDrive/garby_dataset/real/` — real camera photos (min 200 recommended)
   - `/MyDrive/garby_dataset/ai/` — AI-generated images (min 200 recommended)
3. Run all cells in order
4. Download `garby_model.pkl` from Drive and copy to `detection-engine/`

## Dataset recommendations
- **Minimum:** 200 real + 200 AI images
- **Good:** 1000 real + 1000 AI images
- **Best:** 5000+ each, diverse generators (Grok, SDXL, Midjourney, DALL-E 3)
- **Real images:** Mix of DSLR, phone camera, different scenes — avoid screenshots
- **AI images:** Mix of generators for generalisation


## Cell 1: 1. Mount Google Drive


In [ ]:
# Your dataset folder structure in Drive:
#   /MyDrive/garby_dataset/
#       real/        ← real camera photos (JPG, PNG, WEBP)
#       ai/          ← AI-generated images (any generator)
#       holdout/
#           real/    ← held-out real images for final test
#           ai/      ← held-out AI images for final test

from google.colab import drive
drive.mount('/content/drive')

import os
DATASET_PATH = '/content/drive/MyDrive/garby_dataset'
print(f"Dataset path: {DATASET_PATH}")
print(f"Exists: {os.path.exists(DATASET_PATH)}")


## Cell 2: 2. Install dependencies


In [ ]:
# %%capture
import subprocess
subprocess.run(['pip', 'install', 'numpy', 'scipy', 'Pillow', 'scikit-learn',
                'matplotlib', 'seaborn', 'tqdm', 'imbalanced-learn',
                'joblib', 'optuna'], capture_output=True)

print("Dependencies installed.")


## Cell 3: 3. Copy Garby engine files from Drive


In [ ]:
# Upload all garby_layer*.py and garby_orchestrator.py to /MyDrive/garby_engine/

import shutil, sys

ENGINE_SRC = '/content/drive/MyDrive/garby_engine'
ENGINE_DST = '/content/garby_engine'

os.makedirs(ENGINE_DST, exist_ok=True)

engine_files = [
    'garby_layer1_frequency.py',
    'garby_layer2_noise.py',
    'garby_layer3_statistical.py',
    'garby_layer4_semantic.py',
    'garby_layer5_nprdwt.py',
    'garby_orchestrator.py',
]

for f in engine_files:
    src = os.path.join(ENGINE_SRC, f)
    dst = os.path.join(ENGINE_DST, f)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"  ✓ {f}")
    else:
        print(f"  ✗ MISSING: {f} — upload to {ENGINE_SRC}")

sys.path.insert(0, ENGINE_DST)
print("\nEngine files loaded into Python path.")


## Cell 4: 4. Define feature extraction pipeline


In [ ]:
# This runs all 5 layers and extracts a fixed-size feature vector per image.
# We use sub-signal scores, not just the layer ensemble scores, for richer signal.

import numpy as np
from PIL import Image
import traceback

def extract_features(image_path: str) -> dict | None:
    """
    Run all 5 layers and return a dict of sub-signal scores.
    Returns None if the image fails to load or any layer crashes.

    Feature vector (25 values total):
      L1: checkerboard, spectral_falloff, peak_irregularity, channel_asymmetry, ensemble
      L2: prnu, uniformity, kurtosis, local_variance, ensemble
      L3: benford, glcm, histogram, channel_stats, ensemble
      L4: texture_rep, edge_coherence, lighting, contrast, ensemble
      L5: npr, dwt_hh, cross_scale, upsampling, ensemble
    """
    try:
        from garby_layer1_frequency  import analyse as l1
        from garby_layer2_noise      import analyse as l2
        from garby_layer3_statistical import analyse as l3
        from garby_layer4_semantic   import analyse as l4
        from garby_layer5_nprdwt     import analyse as l5

        r1 = l1(image_path)
        r2 = l2(image_path)
        r3 = l3(image_path, layer2_stats=r2.noise_stats)
        r4 = l4(image_path)
        r5 = l5(image_path)

        return {
            # Layer 1 sub-signals
            'l1_checkerboard':    r1.checkerboard_score,
            'l1_spectral':        r1.spectral_falloff_score,
            'l1_peak_irr':        r1.peak_irregularity_score,
            'l1_channel_asym':    r1.channel_asymmetry_score,
            'l1_ensemble':        r1.ensemble_score,
            # Layer 2 sub-signals
            'l2_prnu':            r2.prnu_score,
            'l2_uniformity':      r2.noise_uniformity_score,
            'l2_kurtosis':        r2.residual_kurtosis_score,
            'l2_local_variance':  r2.local_variance_score,
            'l2_ensemble':        r2.ensemble_score,
            # Layer 3 sub-signals
            'l3_benford':         r3.benford_score,
            'l3_glcm':            r3.glcm_score,
            'l3_histogram':       r3.histogram_score,
            'l3_channel_stats':   r3.channel_stats_score,
            'l3_ensemble':        r3.ensemble_score,
            # Layer 4 sub-signals
            'l4_texture_rep':     r4.texture_repetition_score,
            'l4_edge_coherence':  r4.edge_coherence_score,
            'l4_lighting':        r4.lighting_consistency_score,
            'l4_contrast':        r4.local_contrast_score,
            'l4_ensemble':        r4.ensemble_score,
            # Layer 5 sub-signals
            'l5_npr':             r5.npr_score,
            'l5_dwt_hh':          r5.dwt_hh_score,
            'l5_cross_scale':     r5.dwt_cross_scale_score,
            'l5_upsampling':      r5.upsampling_artifact_score,
            'l5_ensemble':        r5.ensemble_score,
        }
    except Exception as e:
        print(f"  ERROR on {os.path.basename(image_path)}: {e}")
        return None


FEATURE_NAMES = [
    'l1_checkerboard','l1_spectral','l1_peak_irr','l1_channel_asym','l1_ensemble',
    'l2_prnu','l2_uniformity','l2_kurtosis','l2_local_variance','l2_ensemble',
    'l3_benford','l3_glcm','l3_histogram','l3_channel_stats','l3_ensemble',
    'l4_texture_rep','l4_edge_coherence','l4_lighting','l4_contrast','l4_ensemble',
    'l5_npr','l5_dwt_hh','l5_cross_scale','l5_upsampling','l5_ensemble',
]
print(f"Feature vector size: {len(FEATURE_NAMES)}")


## Cell 5: 5. Extract features from dataset (takes ~30-60 min for large datasets)


In [ ]:
from tqdm import tqdm

ACCEPTED = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

def collect_image_paths(folder):
    paths = []
    for root, _, files in os.walk(folder):
        for f in files:
            if os.path.splitext(f)[1].lower() in ACCEPTED:
                paths.append(os.path.join(root, f))
    return sorted(paths)

real_dir = os.path.join(DATASET_PATH, 'real')
ai_dir   = os.path.join(DATASET_PATH, 'ai')

real_paths = collect_image_paths(real_dir)
ai_paths   = collect_image_paths(ai_dir)

print(f"Real images found: {len(real_paths)}")
print(f"AI images found:   {len(ai_paths)}")
print(f"Total:             {len(real_paths) + len(ai_paths)}")

if len(real_paths) == 0 or len(ai_paths) == 0:
    raise RuntimeError(
        "No images found! Upload images to:\n"
        f"  {real_dir}\n  {ai_dir}"
    )

# Extract features with progress bar
all_features, all_labels, all_paths = [], [], []

print("\nExtracting features from REAL images...")
for path in tqdm(real_paths):
    feats = extract_features(path)
    if feats:
        all_features.append([feats[k] for k in FEATURE_NAMES])
        all_labels.append(0)   # 0 = REAL
        all_paths.append(path)

print(f"\nExtracting features from AI images...")
for path in tqdm(ai_paths):
    feats = extract_features(path)
    if feats:
        all_features.append([feats[k] for k in FEATURE_NAMES])
        all_labels.append(1)   # 1 = AI
        all_paths.append(path)

X = np.array(all_features, dtype=np.float32)
y = np.array(all_labels,   dtype=np.int32)

print(f"\n{'='*50}")
print(f"Feature matrix shape: {X.shape}")
print(f"Real samples:   {np.sum(y == 0)}")
print(f"AI samples:     {np.sum(y == 1)}")
print(f"Class balance:  {np.sum(y==0)/len(y)*100:.1f}% real / {np.sum(y==1)/len(y)*100:.1f}% AI")
print(f"Failed images:  {len(real_paths)+len(ai_paths) - len(X)}")

# Save to Drive so you don't have to re-extract
import joblib
os.makedirs(os.path.join(DATASET_PATH, 'features'), exist_ok=True)
joblib.dump({'X': X, 'y': y, 'paths': all_paths, 'feature_names': FEATURE_NAMES},
            os.path.join(DATASET_PATH, 'features', 'garby_features.pkl'))
print("\nFeatures saved to Drive.")


## Cell 5b: 5b. [OPTIONAL] Load pre-extracted features (skip Cell 5 if already done)


In [ ]:
# import joblib, numpy as np
# data = joblib.load('/content/drive/MyDrive/garby_dataset/features/garby_features.pkl')
# X, y, all_paths, FEATURE_NAMES = data['X'], data['y'], data['paths'], data['feature_names']
# print(f"Loaded: {X.shape}, Real={np.sum(y==0)}, AI={np.sum(y==1)}")


## Cell 6: 6. Feature distribution analysis


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(5, 5, figsize=(20, 16))
fig.suptitle('Feature Distributions: Real (blue) vs AI (red)', fontsize=14, fontweight='bold')

for idx, (fname, ax) in enumerate(zip(FEATURE_NAMES, axes.flat)):
    real_vals = X[y == 0, idx]
    ai_vals   = X[y == 1, idx]
    ax.hist(real_vals, bins=30, alpha=0.6, color='#2ECC71', label='Real', density=True)
    ax.hist(ai_vals,   bins=30, alpha=0.6, color='#FF3B5C', label='AI',   density=True)
    ax.set_title(fname, fontsize=7)
    ax.set_xlabel('Score', fontsize=6)
    if idx == 0: ax.legend(fontsize=6)

plt.tight_layout()
plt.savefig('/content/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: /content/feature_distributions.png")

# Feature correlation with label
correlations = []
for i, fname in enumerate(FEATURE_NAMES):
    corr = np.corrcoef(X[:, i], y)[0, 1]
    correlations.append((fname, corr))
correlations.sort(key=lambda x: abs(x[1]), reverse=True)

print("\nTop features by correlation with AI label:")
print(f"{'Feature':<25} {'Correlation':>12}")
print('-' * 40)
for fname, corr in correlations:
    bar = '█' * int(abs(corr) * 20)
    direction = '+' if corr > 0 else '-'
    print(f"{fname:<25} {corr:>+.4f}  {direction}{bar}")


## Cell 7: 7. Train / validation / test split


In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold
from sklearn.preprocessing import StandardScaler

# Stratified split: 70% train, 15% val, 15% test
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
train_idx, temp_idx = next(sss.split(X, y))

X_train, y_train = X[train_idx], y[train_idx]
X_temp,  y_temp  = X[temp_idx],  y[temp_idx]

sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
val_idx, test_idx = next(sss2.split(X_temp, y_temp))
X_val,  y_val  = X_temp[val_idx],  y_temp[val_idx]
X_test, y_test = X_temp[test_idx], y_temp[test_idx]

print(f"Train: {len(X_train)} ({np.sum(y_train==0)} real, {np.sum(y_train==1)} AI)")
print(f"Val:   {len(X_val)}   ({np.sum(y_val==0)} real, {np.sum(y_val==1)} AI)")
print(f"Test:  {len(X_test)}  ({np.sum(y_test==0)} real, {np.sum(y_test==1)} AI)")

# Standardise features
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)


## Cell 8: 8. Train classifiers (Logistic Regression, SVM, Random Forest, MLP)


In [ ]:
from sklearn.linear_model    import LogisticRegression
from sklearn.svm             import SVC
from sklearn.ensemble        import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network  import MLPClassifier
from sklearn.calibration     import CalibratedClassifierCV
from sklearn.metrics         import (accuracy_score, roc_auc_score,
                                      classification_report, confusion_matrix)

MODELS = {
    'Logistic Regression': LogisticRegression(
        C=1.0, max_iter=2000, class_weight='balanced', random_state=42),
    'SVM (RBF)': CalibratedClassifierCV(
        SVC(kernel='rbf', C=10, gamma='scale', class_weight='balanced', random_state=42),
        cv=5, method='sigmoid'),
    'Random Forest': RandomForestClassifier(
        n_estimators=300, max_depth=8, min_samples_leaf=5,
        class_weight='balanced', random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        subsample=0.8, random_state=42),
    'MLP': MLPClassifier(
        hidden_layer_sizes=(64, 32), activation='relu',
        max_iter=500, early_stopping=True, validation_fraction=0.1,
        random_state=42),
}

results = {}

print(f"{'Model':<25} {'Acc':>6} {'AUC':>6} {'Prec(AI)':>10} {'Rec(AI)':>9} {'F1(AI)':>8}")
print('─' * 70)

for name, model in MODELS.items():
    model.fit(X_train_s, y_train)
    y_pred = model.predict(X_val_s)
    y_prob = model.predict_proba(X_val_s)[:, 1]

    acc  = accuracy_score(y_val, y_pred)
    auc  = roc_auc_score(y_val, y_prob)
    rep  = classification_report(y_val, y_pred, output_dict=True)
    prec = rep.get('1', {}).get('precision', 0)
    rec  = rep.get('1', {}).get('recall', 0)
    f1   = rep.get('1', {}).get('f1-score', 0)

    results[name] = {'model': model, 'acc': acc, 'auc': auc, 'f1': f1}
    print(f"{name:<25} {acc:>6.3f} {auc:>6.3f} {prec:>10.3f} {rec:>9.3f} {f1:>8.3f}")

best_name = max(results, key=lambda k: results[k]['auc'])
print(f"\nBest model by AUC: {best_name} (AUC={results[best_name]['auc']:.4f})")


## Cell 9: 9. Hyperparameter tuning (Optuna — ~10 min)


In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV

def objective(trial):
    C       = trial.suggest_float('C', 0.1, 100, log=True)
    gamma   = trial.suggest_categorical('gamma', ['scale', 'auto'])
    kernel  = trial.suggest_categorical('kernel', ['rbf', 'poly'])
    degree  = trial.suggest_int('degree', 2, 4) if kernel == 'poly' else 3

    svc = SVC(C=C, gamma=gamma, kernel=kernel, degree=degree,
              class_weight='balanced', probability=True, random_state=42)
    svc.fit(X_train_s, y_train)
    y_prob = svc.predict_proba(X_val_s)[:, 1]
    return roc_auc_score(y_val, y_prob)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=60, show_progress_bar=True)

print(f"\nBest SVM params: {study.best_params}")
print(f"Best val AUC:    {study.best_value:.4f}")

# Train best SVM with calibration
best_p = study.best_params
best_svm = SVC(
    C=best_p['C'], gamma=best_p['gamma'], kernel=best_p['kernel'],
    degree=best_p.get('degree', 3), class_weight='balanced',
    probability=True, random_state=42
)
best_svm.fit(X_train_s, y_train)

# Also tune logistic regression
lr_tuned = LogisticRegression(
    C=best_p.get('C', 1.0),
    class_weight='balanced', max_iter=2000, random_state=42
)
lr_tuned.fit(X_train_s, y_train)


## Cell 10: 10. Final ensemble model


In [ ]:
from sklearn.ensemble import VotingClassifier

# Soft voting ensemble of best models
final_model = VotingClassifier(
    estimators=[
        ('svm', best_svm),
        ('lr',  lr_tuned),
        ('rf',  results['Random Forest']['model']),
    ],
    voting='soft',
    weights=[3, 2, 1],   # SVM gets highest weight, RF least
)
final_model.fit(X_train_s, y_train)

print("Final ensemble trained. Evaluating on validation set...")
y_prob_val = final_model.predict_proba(X_val_s)[:, 1]
y_pred_val = final_model.predict(X_val_s)

print(f"Val Accuracy: {accuracy_score(y_val, y_pred_val):.4f}")
print(f"Val AUC:      {roc_auc_score(y_val, y_prob_val):.4f}")
print()
print(classification_report(y_val, y_pred_val, target_names=['Real', 'AI']))


## Cell 11: 11. Find optimal classification threshold


In [ ]:
from sklearn.metrics import precision_recall_curve, f1_score

y_prob_val = final_model.predict_proba(X_val_s)[:, 1]

thresholds = np.arange(0.20, 0.80, 0.01)
f1_scores, fp_rates, fn_rates = [], [], []

for t in thresholds:
    y_pred_t = (y_prob_val >= t).astype(int)
    f1  = f1_score(y_val, y_pred_t, zero_division=0)
    fp  = np.sum((y_pred_t == 1) & (y_val == 0)) / np.sum(y_val == 0)
    fn  = np.sum((y_pred_t == 0) & (y_val == 1)) / np.sum(y_val == 1)
    f1_scores.append(f1)
    fp_rates.append(fp)
    fn_rates.append(fn)

# Find threshold that maximises F1 while keeping FP rate < 0.15
good_thresholds = [(t, f1, fp, fn)
    for t, f1, fp, fn in zip(thresholds, f1_scores, fp_rates, fn_rates)
    if fp < 0.15]

if good_thresholds:
    best_thresh = max(good_thresholds, key=lambda x: x[1])
else:
    idx_best = np.argmax(f1_scores)
    best_thresh = (thresholds[idx_best], f1_scores[idx_best],
                   fp_rates[idx_best], fn_rates[idx_best])

OPT_THRESHOLD = float(best_thresh[0])
print(f"Optimal threshold: {OPT_THRESHOLD:.2f}")
print(f"  F1:           {best_thresh[1]:.4f}")
print(f"  False pos:    {best_thresh[2]:.2%}  (% of real images wrongly flagged as AI)")
print(f"  False neg:    {best_thresh[3]:.2%}  (% of AI images missed)")

# Plot threshold analysis
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(thresholds, f1_scores, color='#2ECC71', lw=2, label='F1 Score')
ax1.axvline(OPT_THRESHOLD, color='#FF3B5C', linestyle='--', label=f'Optimal: {OPT_THRESHOLD:.2f}')
ax1.set_xlabel('Threshold'); ax1.set_ylabel('F1'); ax1.legend()
ax1.set_title('F1 Score vs Classification Threshold')

ax2.plot(thresholds, fp_rates, color='#FF3B5C', lw=2, label='False Positive Rate')
ax2.plot(thresholds, fn_rates, color='#F59E0B', lw=2, label='False Negative Rate')
ax2.axvline(OPT_THRESHOLD, color='#07081A', linestyle='--', label=f'Optimal: {OPT_THRESHOLD:.2f}')
ax2.set_xlabel('Threshold'); ax2.set_ylabel('Rate'); ax2.legend()
ax2.set_title('Error Rates vs Threshold')

plt.tight_layout()
plt.savefig('/content/threshold_analysis.png', dpi=150)
plt.show()


## Cell 12: 12. Final test set evaluation (run ONCE at the end)


In [ ]:
print("=" * 60)
print("FINAL TEST SET EVALUATION")
print("=" * 60)

y_prob_test = final_model.predict_proba(X_test_s)[:, 1]
y_pred_test = (y_prob_test >= OPT_THRESHOLD).astype(int)

acc = accuracy_score(y_test, y_pred_test)
auc = roc_auc_score(y_test, y_prob_test)

print(f"Test Accuracy:  {acc:.4f}")
print(f"Test AUC:       {auc:.4f}")
print()
print(classification_report(y_test, y_pred_test, target_names=['Real', 'AI']))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred_test)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
    xticklabels=['Pred Real','Pred AI'], yticklabels=['True Real','True AI'])
axes[0].set_title(f'Confusion Matrix (threshold={OPT_THRESHOLD:.2f})')

# ROC curve
from sklearn.metrics import roc_curve
fpr, tpr, _ = roc_curve(y_test, y_prob_test)
axes[1].plot(fpr, tpr, color='#2ECC71', lw=2, label=f'AUC = {auc:.4f}')
axes[1].plot([0,1],[0,1], 'k--', lw=1)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate (Recall)')
axes[1].set_title('ROC Curve')
axes[1].legend()

plt.tight_layout()
plt.savefig('/content/evaluation.png', dpi=150)
plt.show()

# Per-class analysis
tn, fp, fn, tp = cm.ravel()
print(f"\nTrue Positives  (AI correctly detected):  {tp}")
print(f"True Negatives  (Real correctly passed):  {tn}")
print(f"False Positives (Real wrongly flagged):   {fp} ({fp/(tn+fp)*100:.1f}%)")
print(f"False Negatives (AI missed):              {fn} ({fn/(tp+fn)*100:.1f}%)")


## Cell 13: 13. Feature importance (what the model learned)


In [ ]:
# From Random Forest (most interpretable)
rf_model = results['Random Forest']['model']
importances = rf_model.feature_importances_

feat_imp = sorted(zip(FEATURE_NAMES, importances), key=lambda x: x[1], reverse=True)

fig, ax = plt.subplots(figsize=(10, 8))
names = [f[0] for f in feat_imp]
vals  = [f[1] for f in feat_imp]
colours = ['#FF3B5C' if 'l4' in n or 'l3' in n or 'l2' in n
           else '#2ECC71' if 'l5' in n or 'l1' in n
           else '#5B21D6' for n in names]
ax.barh(names[::-1], vals[::-1], color=colours[::-1])
ax.set_xlabel('Feature Importance')
ax.set_title('Garby Engine — Feature Importance\n(Red=L2/L3/L4, Green=L1/L5)')
plt.tight_layout()
plt.savefig('/content/feature_importance.png', dpi=150)
plt.show()

print("Top 10 most important features:")
for name, imp in feat_imp[:10]:
    bar = '█' * int(imp * 200)
    print(f"  {name:<25} {imp:.4f}  {bar}")


## Cell 14: 14. Export model → garby_model.pkl


In [ ]:
import json, joblib, datetime

output_dir = '/content/drive/MyDrive/garby_trained_model'
os.makedirs(output_dir, exist_ok=True)

# Save sklearn model + scaler + metadata
model_bundle = {
    'model':         final_model,
    'scaler':        scaler,
    'feature_names': FEATURE_NAMES,
    'threshold':     OPT_THRESHOLD,
    'trained_at':    datetime.datetime.utcnow().isoformat(),
    'train_size':    len(X_train),
    'val_auc':       results[best_name]['auc'],
    'test_auc':      float(auc),
    'test_accuracy': float(acc),
    'class_map':     {0: 'REAL', 1: 'AI_GENERATED'},
}

model_path = os.path.join(output_dir, 'garby_model.pkl')
joblib.dump(model_bundle, model_path, compress=3)
print(f"Model saved: {model_path}")

# Also export training metadata as JSON (human-readable)
meta = {k: v for k, v in model_bundle.items() if k not in ('model', 'scaler')}
with open(os.path.join(output_dir, 'model_metadata.json'), 'w') as f:
    json.dump(meta, f, indent=2)
print(f"Metadata saved.")

# Export learned layer weights for integration with existing engine
# This tells the orchestrator how to re-weight the 5 layers
lr_model = lr_tuned
layer_feature_groups = {
    'layer1': [i for i, n in enumerate(FEATURE_NAMES) if n.startswith('l1_')],
    'layer2': [i for i, n in enumerate(FEATURE_NAMES) if n.startswith('l2_')],
    'layer3': [i for i, n in enumerate(FEATURE_NAMES) if n.startswith('l3_')],
    'layer4': [i for i, n in enumerate(FEATURE_NAMES) if n.startswith('l4_')],
    'layer5': [i for i, n in enumerate(FEATURE_NAMES) if n.startswith('l5_')],
}

# Compute mean absolute coefficient per layer (proxy for importance)
abs_coef = np.abs(lr_model.coef_[0])
layer_weights_raw = {}
for layer, indices in layer_feature_groups.items():
    layer_weights_raw[layer] = float(np.mean(abs_coef[indices]))

# Normalise to sum to 1
total_w = sum(layer_weights_raw.values())
learned_layer_weights = {k: round(v/total_w, 4) for k, v in layer_weights_raw.items()}

print(f"\nLearned layer weights (from logistic regression coefficients):")
for layer, w in learned_layer_weights.items():
    bar = '█' * int(w * 100)
    print(f"  {layer}: {w:.4f}  {bar}")

weights_path = os.path.join(output_dir, 'learned_weights.json')
with open(weights_path, 'w') as f:
    json.dump({
        'layer_weights': learned_layer_weights,
        'threshold':     OPT_THRESHOLD,
        'feature_importance': dict(feat_imp),
    }, f, indent=2)
print(f"\nWeights saved: {weights_path}")
print("\nDownload garby_model.pkl and deploy to detection-engine/")


## Cell 15: 15. Generate orchestrator integration patch


In [ ]:
patch_code = f'''
# ── Trained model integration for garby_orchestrator.py ──────────────────────
# Replace the hand-tuned LAYER_WEIGHTS and determine_final_verdict with
# the trained model. Add this to garby_orchestrator.py and call
# detect_with_model() instead of detect() when model file is present.

import joblib, os
import numpy as np

_MODEL_PATH = os.path.join(os.path.dirname(__file__), 'garby_model.pkl')
_model_bundle = None

def _load_model():
    global _model_bundle
    if _model_bundle is None and os.path.exists(_MODEL_PATH):
        _model_bundle = joblib.load(_MODEL_PATH)
        print(f"[GarbyEngine] Trained model loaded (AUC={{_model_bundle.get(\\'test_auc\\', 0):.4f}})")
    return _model_bundle

def detect_trained(image_path: str, verbose: bool = False) -> GarbyResult:
    """
    Run detection using the trained meta-classifier.
    Falls back to rule-based detect() if model file not present.
    """
    bundle = _load_model()
    if bundle is None:
        if verbose: print("[GarbyEngine] No trained model found — using rule-based detection")
        return detect(image_path, verbose=verbose)

    import time
    start = time.perf_counter()
    filename = os.path.basename(image_path)

    # Run all 5 layers
    r1 = _l1(image_path)
    r2 = _l2(image_path)
    r3 = _l3(image_path, layer2_stats=r2.noise_stats)
    r4 = _l4(image_path)
    r5 = _l5(image_path)

    # Build feature vector
    FEATURE_NAMES = bundle[\\'feature_names\\']
    feat_map = {{
        \\'l1_checkerboard\\': r1.checkerboard_score,
        \\'l1_spectral\\':     r1.spectral_falloff_score,
        \\'l1_peak_irr\\':     r1.peak_irregularity_score,
        \\'l1_channel_asym\\': r1.channel_asymmetry_score,
        \\'l1_ensemble\\':     r1.ensemble_score,
        \\'l2_prnu\\':         r2.prnu_score,
        \\'l2_uniformity\\':   r2.noise_uniformity_score,
        \\'l2_kurtosis\\':     r2.residual_kurtosis_score,
        \\'l2_local_variance\\': r2.local_variance_score,
        \\'l2_ensemble\\':     r2.ensemble_score,
        \\'l3_benford\\':      r3.benford_score,
        \\'l3_glcm\\':         r3.glcm_score,
        \\'l3_histogram\\':    r3.histogram_score,
        \\'l3_channel_stats\\': r3.channel_stats_score,
        \\'l3_ensemble\\':     r3.ensemble_score,
        \\'l4_texture_rep\\':  r4.texture_repetition_score,
        \\'l4_edge_coherence\\': r4.edge_coherence_score,
        \\'l4_lighting\\':     r4.lighting_consistency_score,
        \\'l4_contrast\\':     r4.local_contrast_score,
        \\'l4_ensemble\\':     r4.ensemble_score,
        \\'l5_npr\\':          r5.npr_score,
        \\'l5_dwt_hh\\':       r5.dwt_hh_score,
        \\'l5_cross_scale\\':  r5.dwt_cross_scale_score,
        \\'l5_upsampling\\':   r5.upsampling_artifact_score,
        \\'l5_ensemble\\':     r5.ensemble_score,
    }}
    X = np.array([[feat_map[k] for k in FEATURE_NAMES]], dtype=np.float32)

    # Scale and predict
    X_s     = bundle[\\'scaler\\'].transform(X)
    ai_prob = float(bundle[\\'model\\'].predict_proba(X_s)[0, 1])
    threshold = bundle.get(\\'threshold\\', 0.45)

    if ai_prob >= threshold:
        verdict, confidence = "AI-Generated", "High" if ai_prob > 0.70 else "Medium"
    elif ai_prob >= threshold * 0.75:
        verdict, confidence = "Inconclusive", "Low"
    else:
        verdict, confidence = "Likely Real", "High" if ai_prob < 0.20 else "Medium"

    elapsed_ms = round((time.perf_counter() - start) * 1000, 1)

    return GarbyResult(
        image_path=image_path, filename=filename,
        verdict=verdict, confidence=confidence,
        ai_probability=round(ai_prob, 4),
        confidence_pct=int(round(ai_prob * 100 if "AI" in verdict else (1 - ai_prob) * 100)),
        layer1_score=round(r1.ai_probability, 4),
        layer2_score=round(r2.ai_probability, 4),
        layer3_score=round(r3.ai_probability, 4),
        layer4_score=round(r4.ai_probability, 4),
        layer5_score=round(r5.ai_probability, 4),
        ensemble_score=round(ai_prob, 4),
        layers_agreeing=sum(1 for v in [r1.verdict, r2.verdict, r3.verdict, r4.verdict, r5.verdict] if "AI" in v),
        signals=merge_signals(r1, r2, r3, r4, r5),
        findings=r4.findings,
        processing_time_ms=elapsed_ms,
        layer_results={{"layer1": r1, "layer2": r2, "layer3": r3, "layer4": r4, "layer5": r5}},
    )
'''

patch_path = os.path.join(output_dir, 'orchestrator_patch.py')
with open(patch_path, 'w') as f:
    f.write(patch_code)
print(f"Integration patch saved: {patch_path}")
print("\n✅ Training complete! Files saved to Google Drive:")
print(f"   {output_dir}/")
print(f"     garby_model.pkl          ← main model, copy to detection-engine/")
print(f"     learned_weights.json     ← human-readable layer weights")
print(f"     orchestrator_patch.py    ← integration code for garby_orchestrator.py")
print(f"     model_metadata.json      ← training stats")
